In [ ]:
# 실습 준비 — 15주차 기말고사
# 이 셀을 먼저 한 번 실행하세요. 데이터가 없으면 아래 셀들이 전부 실패합니다.
import os, pathlib, urllib.request

BASE = "https://raw.githubusercontent.com/aprilslab/statistics-lab/main/data/"
FILES = ["final_scores.csv"]

pathlib.Path("data").mkdir(exist_ok=True)
for name in FILES:
    for dest in (pathlib.Path(name), pathlib.Path("data") / name):
        if not dest.exists():
            urllib.request.urlretrieve(BASE + name, dest)

# '../data/x.csv' 로 읽는 노트북 대응 — 상위 폴더에도 같은 data/ 를 걸어둔다.
# 절대경로(/data)로 박으면 cwd 가 /content 가 아닐 때 깨지므로 상대경로로 건다.
try:
    parent = pathlib.Path("..") / "data"
    if not parent.exists():
        os.symlink(pathlib.Path("data").resolve(), parent)
except OSError:
    pass

print("준비 완료:", ", ".join(FILES) if FILES else "(내려받을 데이터 없음)")


# 기말고사 대체과제

`final_scores.csv`는 한 학교 학생중 50명의 최종성적과 공부시간, 출석률, SNS사용시간, 캡스톤점수의 데이터를 수집한 데이터이다.

* 해석 등의 텍스트를 서술할 때는 마크다운 셀을 이용합니다.
* 코드셀이나 마크다운 셀은 필요한만큼 추가하셔도 좋습니다.
* 제출은 `기말고사_{본인이름}.ipynb`이름으로 제출합니다.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

df = pd.read_csv('final_scores.csv')
df.head()

,공부시간,출석률,SNS사용시간,캡스톤점수,최종성적
0,4.22,0.870,1.57,76.9,75.9
1,5.49,0.915,2.07,77.0,86.7
2,6.96,0.826,3.07,84.9,82.1
3,6.94,0.849,2.37,87.3,81.1
4,3.74,0.829,4.41,69.9,68.6


## 1. 신뢰구간 추청 [10점]

`최종성적` 데이터의 모평균의 95% 신뢰구간을 구하세요.
* 표본분산을 구합니다.
* 모분산은 모른다고 가정하고 95% 신뢰구간을 구합니다.
* 결과를 해석합니다.

In [3]:
scores = df["최종성적"]

In [15]:
n = len(scores)
sample_mean = scores.mean()
sample_var = scores.var(ddof=1)   # ddof=1 이면 표본분산
sample_mean, sample_var

(np.float64(79.42200000000001), np.float64(68.47440408163267))

In [14]:
rv = stats.t(n-1)
lcl = sample_mean - rv.isf(0.025) * np.sqrt(sample_var/n)
ucl = sample_mean - rv.isf(0.975) * np.sqrt(sample_var/n)
lcl, ucl

(np.float64(77.07029198649535), np.float64(81.77370801350467))

## 2. 가설검정 [10점]
다음 상황에서 통계적 가설검정을 수행하세요. (유의수준 α = 0.05)
* 모분산이 64라고 알려져 있다고 가정합니다.
* '최종성적 모평균이 75점이다'라는 가설을 검정합니다.
* `귀무가설` `대립가설`을 설정합니다.
* 유의수준 5%의 양측 검정을 시행합니다.
* 귀무가성을 채택/기각 합니다.
* 결과를 해석합니다.

In [16]:
# 귀무가설 H0 = 모평균이 75점이다.
# 대립가설 H1 = 모평균이 75가 아니다.
mu0 = 75
p_var = 64
n = len(scores)
s_mean = scores.mean()
s_mean

np.float64(79.42200000000001)

In [18]:
rv = stats.norm()
lcl = sample_mean - rv.isf(0.025) * np.sqrt(p_var/n)
ucl = sample_mean - rv.isf(0.975) * np.sqrt(p_var/n)
lcl, ucl

(np.float64(77.20455388104052), np.float64(81.6394461189595))

In [ ]:
# 귀무가설 기각

## 3. 상관분석 [10점]
`공부시간`, `출석률`, `SNS사용시간`, `캡스톤점수`, `최종성적` 다섯 변수의 상관관계를 분석하세요.
* 변수들의 상관계수 행렬을 구합니다.
* `최종성적`변수와 나머지 변수의 관계가 어떤 관계를 보이는지 산점도로 시각화 하세요.
* `최종성적`변수와 나머지 변수가 어떤 상관관계를 보이는지 간단히 해석하세요.

In [19]:
df.corr()

,공부시간,출석률,SNS사용시간,캡스톤점수,최종성적
공부시간,1.000000,0.553157,-0.735128,0.841072,0.774627
출석률,0.553157,1.000000,-0.371226,0.509123,0.437992
SNS사용시간,-0.735128,-0.371226,1.000000,-0.647690,-0.781501
캡스톤점수,0.841072,0.509123,-0.647690,1.000000,0.726798
최종성적,0.774627,0.437992,-0.781501,0.726798,1.000000


## 4. 회귀분석 [10점]
* 반응변수: `최종성적`
* 설명변수: `공부시간`, `출석률`, `SNS사용시간`, `캡스톤점수`
* 설명변수로 반응변수를 예측하는 회귀모형을 만듭니다.
* 만든 회귀 모형을 비교하여 가장 적절하다고 생각하는 모형을 선택합니다.
* 선택 이유를 서술하세요. (상관계수, summary의 지표 등을 근거로 언급할 것)